# Lab 1 · Build a RAG pipeline (free Kaggle T4)

**Layer 3 · Data.** We ground **Qwen2.5-3B-Instruct** — the same model as the
Models labs — in a small **Kubernetes / SRE knowledge base** it has never seen:
the internal runbooks, cluster policy, on-call rules, and a postmortem for a
fictional platform team ("Globex"). This is the *vector-DB-of-runbooks* from the
SRE-assistant case study, in miniature.

The arc of this lab is one ops question asked **twice**:

1. **Without retrieval** → the model knows generic k8s but not *your* runbook, so
   it *hallucinates* a confident, wrong remediation.
2. **With retrieval** → we fetch the relevant runbook chunks and the answer
   becomes correct **and cited**.

Pipeline: `runbooks → chunk → embed → FAISS → retrieve → augment → generate`.

**Settings:** Accelerator = **GPU (T4)**, Internet = **On**. Then **Run All**.

Outputs written to `/kaggle/working` (`index.faiss`, `chunks.jsonl`, `qa.jsonl`)
feed **Lab 2** — *Save Version* when done.

## 0 · Install dependencies

Kaggle ships PyTorch + Transformers. We add FAISS (the vector store) and
sentence-transformers (the embedder). This is the only network-heavy step.

In [ ]:
!pip install -q faiss-cpu sentence-transformers

import torch, faiss, json, os, textwrap
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('faiss', faiss.__version__)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORK = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)

## 1 · The corpus (Phase 1 · Sources)

A RAG demo only works if the model *can't* already know the answer. Generic
Kubernetes facts (what `CrashLoopBackOff` means) are in the model's training data —
so our corpus blends those with **org-specific** details no model could know:
Globex's namespaces, thresholds, channels, and a dated incident. That org layer is
exactly the *private data* the Data layer exists to supplement.

Each document carries **metadata** (id, title) — in production this is also where
the ACL / tenant tag and timestamp live (Phase 1 of the README).

In [ ]:
DOCS = [
  {'id': 'k8s-pod-failures', 'title': 'Runbook: pod failures',
   'text': (
     'A pod in CrashLoopBackOff is starting and crashing repeatedly; the kubelet backs off restarts up to 5 minutes. '
     'An OOMKilled pod was terminated for exceeding its memory limit and shows exit code 137. '
     'ImagePullBackOff means the image cannot be pulled, usually a bad tag or missing registry credentials. '
     'Globex runbook: for OOMKilled in the payments namespace, raise the memory limit to 2Gi, redeploy, and page the #sre-payments channel.'
   )},
  {'id': 'k8s-probes', 'title': 'Runbook: health probes',
   'text': (
     'A liveness probe restarts a container that is alive but stuck; a readiness probe pulls a pod out of the Service endpoints until it can serve traffic. '
     'A startup probe protects slow-starting containers so the liveness probe does not kill them too early. '
     'Globex default: liveness is an httpGet on /healthz every 10 seconds with a failure threshold of 3; readiness is an httpGet on /ready.'
   )},
  {'id': 'k8s-scaling', 'title': 'Runbook: autoscaling and resources',
   'text': (
     'Requests are what the scheduler reserves; limits are the hard ceiling the kubelet enforces, and exceeding the memory limit triggers an OOMKill. '
     'The HorizontalPodAutoscaler scales replicas on CPU or custom metrics. '
     'The Globex web tier scales between 3 and 30 replicas at a target of 70 percent CPU. '
     'The cluster autoscaler adds nodes from the burst node pool when pods stay unschedulable for more than 2 minutes.'
   )},
  {'id': 'k8s-oncall', 'title': 'Runbook: on-call and severity',
   'text': (
     'Globex uses three severity levels: SEV1 is a full outage, SEV2 is degraded service, SEV3 is a minor issue. '
     'A SEV1 pages the primary on-call immediately and the secondary after 10 minutes. '
     'The automated alert threshold is 3 pod restarts within 10 minutes. '
     'On-call rotations are weekly and hand over every Monday at 10:00.'
   )},
  {'id': 'k8s-deploy', 'title': 'Runbook: deployments and rollback',
   'text': (
     'Globex deploys with a rolling update: max surge 25 percent, max unavailable 0. '
     'Every production deploy must pass a canary that takes 5 percent of traffic for 15 minutes before full rollout. '
     'To roll back, run kubectl rollout undo on the deployment, which reverts to the previous ReplicaSet. '
     'Deploys are frozen on Fridays after 14:00 and during any SEV1 or SEV2 incident.'
   )},
  {'id': 'incident-2026-04', 'title': 'Postmortem: payments outage 12 Apr 2026',
   'text': (
     'On 12 April 2026 the payments-api entered CrashLoopBackOff after release v2.3.1 introduced a memory leak. '
     'Pods were OOMKilled with exit code 137 about every 4 minutes, tripping the restart alert. '
     'The on-call raised the memory limit to 2Gi as a stopgap and rolled back to v2.3.0 with kubectl rollout undo. '
     'Root cause: an unbounded in-memory cache; the fix shipped in v2.3.2 with a cache size cap.'
   )},
]
print(f'{len(DOCS)} documents, {sum(len(d["text"].split()) for d in DOCS)} words total')

## 2 · Chunking (Phase 2)

We split each document into fixed-size, **token**-counted chunks with overlap so a
sentence cut at a boundary still survives in the neighbour. Our docs are short, so
a small chunk size keeps each chunk on one topic — better retrieval precision.

We count tokens with the **LLM's own tokenizer** so "chunk size" matches what the
model will actually read.

In [ ]:
from transformers import AutoTokenizer
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL_ID)

def chunk_text(text, size=64, overlap=12):
    ids = tok.encode(text, add_special_tokens=False)
    step = size - overlap
    out = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + size]
        if not piece:
            break
        out.append(tok.decode(piece))
        if start + size >= len(ids):
            break
    return out

CHUNK_SIZE, OVERLAP = 64, 12
chunks = []
for d in DOCS:
    for j, c in enumerate(chunk_text(d['text'], CHUNK_SIZE, OVERLAP)):
        chunks.append({'chunk_id': f"{d['id']}::{j}", 'doc_id': d['id'],
                       'title': d['title'], 'text': c})
print(f'{len(chunks)} chunks from {len(DOCS)} docs (size={CHUNK_SIZE}, overlap={OVERLAP} tokens)')
print('\nexample chunk:\n', chunks[2]['text'])

## 3 · Embeddings (Phase 3)

We encode every chunk into a 384-dim vector with **`bge-small-en-v1.5`** — a small,
fast, CPU-friendly embedding model that is *separate* from the LLM.

Two `bge` specifics worth knowing:
- queries get an **instruction prefix** (`Represent this sentence...`); documents
  do not. Using the wrong side is a classic silent bug.
- we **L2-normalize** so an inner-product index computes **cosine** similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMB_ID = 'BAAI/bge-small-en-v1.5'
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder = SentenceTransformer(EMB_ID, device=DEVICE)

def embed_docs(texts):
    return embedder.encode(texts, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)

def embed_queries(texts):
    return embedder.encode([QUERY_PREFIX + t for t in texts], normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)

doc_vecs = embed_docs([c['text'] for c in chunks]).astype('float32')
print('embedded', doc_vecs.shape, '(chunks x dims)')

## 4 · Vector store (Phase 4)

With only a few hundred chunks we use a **flat** FAISS index — *exact* search, no
approximation needed. `IndexFlatIP` on normalized vectors = cosine similarity. At
millions of chunks you would switch to **HNSW** or **IVF** (the README table); the
retrieval code below would not change.

In [ ]:
dim = doc_vecs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_vecs)
print('index holds', index.ntotal, 'vectors of dim', dim)

def retrieve(query, k=3):
    qv = embed_queries([query]).astype('float32')
    scores, idx = index.search(qv, k)
    hits = []
    for score, i in zip(scores[0], idx[0]):
        c = chunks[int(i)]
        hits.append({**c, 'score': float(score)})
    return hits

for h in retrieve('what do I do when a payments pod runs out of memory?', k=3):
    print(f"{h['score']:.3f}  {h['chunk_id']:<22} {h['text'][:70]}...")

## 5 · The LLM

Load Qwen2.5-3B-Instruct in **fp16** (the T4 has no bf16). ~6 GB of VRAM — the
FAISS index lives in RAM, so RAG adds almost nothing on top.

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             device_map='auto')
model.eval()

def generate(messages, max_new_tokens=200):
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

if torch.cuda.is_available():
    print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 6 · Ask **without** retrieval — watch it hallucinate

The model knows generic Kubernetes, but it has never seen Globex's runbook — so the
org-specific parts (the exact limit, the channel to page) are invented. Note how
confident the wrong answer is — this is exactly the failure RAG fixes.

In [ ]:
QUESTION = 'In the Globex SRE runbook, when a pod in the payments namespace is OOMKilled, what memory limit should I set, and which channel do I page?'

def ask_without_rag(question):
    return generate([
        {'role': 'system', 'content': 'You are a helpful SRE assistant.'},
        {'role': 'user', 'content': question},
    ])

print(textwrap.fill(ask_without_rag(QUESTION), 90))

## 7 · Ask **with** retrieval (Phases 5–6 · RAG)

Retrieve the top-k chunks, stitch them into the prompt as **CONTEXT**, and instruct
the model to answer *only* from it and to **cite** the chunk ids. Same model, same
question — grounded this time (2Gi, #sre-payments).

In [ ]:
SYSTEM_RAG = ('You are an SRE assistant. Answer using ONLY the runbook context below. '
              'Cite the chunk ids you used in square brackets, e.g. [k8s-pod-failures::0]. '
              "If the answer is not in the context, say you don't know.")

def build_prompt(question, hits):
    ctx = '\n\n'.join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    return [
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user', 'content': f'CONTEXT:\n{ctx}\n\nQUESTION: {question}'},
    ]

def ask_with_rag(question, k=3):
    hits = retrieve(question, k=k)
    answer = generate(build_prompt(question, hits))
    return answer, hits

answer, hits = ask_with_rag(QUESTION, k=3)
print('RETRIEVED:')
for h in hits:
    print(f"  {h['score']:.3f}  [{h['chunk_id']}]")
print('\nANSWER:\n', textwrap.fill(answer, 90))

## 8 · The "I don't know" guardrail

A good RAG system **refuses** when the runbooks don't cover the question, instead
of inventing an answer. Ask something the corpus never mentions.

In [ ]:
off_topic = 'How much does a Globex enterprise Kubernetes support contract cost per year?'
ans, hits = ask_with_rag(off_topic, k=3)
print('top score:', round(hits[0]['score'], 3))
print(textwrap.fill(ans, 90))

## 9 · Save outputs for Lab 2

Persist the index, the chunks, and a small **gold Q&A set** (question → the runbook
that answers it). Lab 2 reads these to score retrieval and faithfulness. Click
**Save Version** so this output can be attached as Lab 2's input.

In [ ]:
QA = [
  {'question': 'For an OOMKilled pod in the payments namespace, what memory limit should I set?', 'gold_doc': 'k8s-pod-failures'},
  {'question': 'What exit code does an OOMKilled pod show?', 'gold_doc': 'k8s-pod-failures'},
  {'question': 'What is the automated alert threshold for pod restarts at Globex?', 'gold_doc': 'k8s-oncall'},
  {'question': 'How do I roll back a bad Kubernetes deployment at Globex?', 'gold_doc': 'k8s-deploy'},
  {'question': 'How much traffic does a Globex canary take, and for how long?', 'gold_doc': 'k8s-deploy'},
  {'question': 'What caused the payments outage on 12 April 2026?', 'gold_doc': 'incident-2026-04'},
  {'question': 'What are the three Globex severity levels?', 'gold_doc': 'k8s-oncall'},
  {'question': 'What is the Globex default liveness probe configuration?', 'gold_doc': 'k8s-probes'},
]

faiss.write_index(index, f'{WORK}/index.faiss')
with open(f'{WORK}/chunks.jsonl', 'w') as f:
    for c in chunks:
        f.write(json.dumps(c) + '\n')
with open(f'{WORK}/qa.jsonl', 'w') as f:
    for q in QA:
        f.write(json.dumps(q) + '\n')
with open(f'{WORK}/meta.json', 'w') as f:
    json.dump({'emb_id': EMB_ID, 'model_id': MODEL_ID, 'chunk_size': CHUNK_SIZE,
               'overlap': OVERLAP, 'query_prefix': QUERY_PREFIX, 'n_chunks': len(chunks)}, f)
print('saved:', os.listdir(WORK))

## Takeaways

- **Same model, two answers.** Without context Qwen invents Globex's limit and
  channel; with the retrieved runbook it is correct **and cited**. The model
  didn't get smarter — it got *grounded* in your ops knowledge.
- This is the **SRE-assistant pattern** in miniature: a vector DB of runbooks +
  incidents, queried in natural language. Generic k8s the model already knows; the
  *org-specific* layer is what RAG supplies.
- **The embedder is not the LLM**, and documents/queries must use the **same**
  embedder (with `bge`'s query prefix) — the most common silent RAG bug.
- A real system must also **refuse** when the runbooks can't answer (cell 8).

Next: **[Lab 2](lab2_rageval_kaggle.ipynb)** — we stop trusting "it looks right"
and put **numbers** on retrieval and faithfulness, then sweep chunk-size, top-k,
and a reranker.